# Natural Language Processing (NLP)

NLP turns **raw text into numbers** a model can learn from. Spam filters, search ranking, sentiment dashboards, translation, chat — all of it starts with the same staircase: preprocess → vectorize → model. This note walks that staircase on a **real dataset**, from Bag-of-Words through TF-IDF to word embeddings, and ends at the door of Transformers (Level 7).

> **Prerequisites:** Level 1 Python/pandas, Level 2 Section 3 (probability helps for Naive Bayes), and the [ML Metrics](https://github.com/AdilShamim8/ML-Roadmap-and-Notes/tree/main/2.%20Gaining%20Conceptual%20depth/7.%20Machine%20Learning%20Metrics) notes for precision/recall/F1 — this task is imbalanced, so we report F1, not accuracy.


## 1. The NLP Staircase

| Rung | Idea | Era | Still used? |
|---|---|---|---|
| **Preprocessing** | Clean, tokenize, normalize | always | Yes — every pipeline |
| **Bag-of-Words / TF-IDF** | Count words; downweight common ones | 1950s–2010s | Yes — fast, cheap, interpretable; strong on short texts |
| **Word embeddings** (Word2Vec, GloVe) | Dense vectors; similar meaning → similar vector | 2013– | As features; largely superseded by contextual embeddings |
| **RNN / LSTM** | Read sequence left-to-right, keep a memory | 2014–2017 | Legacy; long-range dependencies were painful |
| **Transformers** | Self-attention: every token attends to every token | 2017– | The backbone of everything modern |
| **LLMs** (Level 7) | Pretrained transformers at extreme scale; prompt or fine-tune | 2018– | The 2026 default for most tasks |

Classic NLP is **not** obsolete: a BOW + Naive Bayes spam filter trains in milliseconds on a laptop, needs no GPU, and is fully explainable — for millions of messages a day that matters. Learn the staircase bottom-up because the top rungs assume you know what a "token" and an "idf weight" are.


## 2. The Dataset: A Real SMS Spam Collection

We use the **UCI SMS Spam Collection** (Almeida & Hidalgo, 2011; publicly released for research): **5,574 real text messages**, 4,827 legitimate (*ham*) and 747 spam — a 13.4% spam rate, genuinely imbalanced, exactly like production. The file ships beside this notebook (`SMSSpamCollection.tsv`, tab-separated: `label<TAB>message`), so everything below runs offline.


In [1]:
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

df = pd.read_csv("SMSSpamCollection.tsv", sep="\t", names=["label", "message"])
print(f"{len(df):,} messages — {df.label.value_counts().to_dict()}")
print(f"spam rate: {(df.label == 'spam').mean():.1%}")
print(f"avg characters: ham={df[df.label=='ham'].message.str.len().mean():.0f}, "
      f"spam={df[df.label=='spam'].message.str.len().mean():.0f}\n")
df.sample(3, random_state=1)

5,574 messages — {'ham': 4827, 'spam': 747}
spam rate: 13.4%
avg characters: ham=72, spam=139



,label,message
1447,ham,Looks like u wil b getting a headstart im leav...
2032,ham,"I noe la... U wana pei bf oso rite... K lor, o..."
4432,ham,2mro i am not coming to gym machan. Goodnight.


## 3. Preprocessing: Making Text Learnable

Raw text is messy — case, punctuation, slang. The standard cleaning pass:

1. **Lowercase** — "Free" and "free" should be one token.
2. **Tokenize** — split into words (rule-based or model-based; `simple_preprocess` below is a solid rule-based tokenizer).
3. **Remove stopwords** — high-frequency, low-content words (*the*, *is*, *at*...).
4. **Stem or lemmatize** (optional) — *winning/wins/won* → one stem. Stemming is crude rule-chopping; lemmatization uses a dictionary. We skip it here — short spam texts need every distinctive character, and stems like `noki` hurt more than help.

> **A real gotcha:** scikit-learn's stop-word list contains **318 words** including *call* and *txt*-adjacent words like *bill*... no, but it *does* include **call** — a genuinely discriminative word for spam. Inspect the stop list before trusting it blindly for your task. (We keep it here because the damage is small, and we get an honest lesson.)


In [2]:
from gensim.utils import simple_preprocess

msg = df.loc[df.label == "spam", "message"].iloc[0]
print(f"raw:      {msg[:90]}...")
print(f"tokens:   {simple_preprocess(msg)[:12]} ...")
print(f"stopword examples: {sorted(list(ENGLISH_STOP_WORDS))[:8]} ...  ({len(ENGLISH_STOP_WORDS)} total)")
print(f"'call' in stop list: {'call' in ENGLISH_STOP_WORDS}")

raw:      Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to re...
tokens:   ['free', 'entry', 'in', 'wkly', 'comp', 'to', 'win', 'fa', 'cup', 'final', 'tkts', 'st'] ...
stopword examples: ['a', 'about', 'above', 'across', 'after', 'afterwards', 'again', 'against'] ...  (318 total)
'call' in stop list: True


## 4. Bag-of-Words: Counting Is a Baseline That Punches

**Bag-of-Words (BOW)** builds a vocabulary of all unique terms, then represents each message as a row of counts — word order is discarded, only membership and frequency remain. `CountVectorizer` does it in one line; with `stop_words` and `lowercase=True` it fuses the preprocessing of Section 3 into the vectorizer itself (the correct pattern — put it inside the pipeline, Level 3 §6, so train/test splits never leak).

Then the classic pairing: **Multinomial Naive Bayes** — with BOW counts it is the textbook spam filter (Bayes' theorem with a word-independence assumption, Level 2 §3 gives the probability behind it).


In [3]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import cross_val_score

y = (df.label == "spam").astype(int)
X = df.message

bow_pipe = make_pipeline(
    CountVectorizer(stop_words=list(ENGLISH_STOP_WORDS), lowercase=True),
    MultinomialNB(),
)
f1_bow = cross_val_score(bow_pipe, X, y, cv=5, scoring="f1")
print(f"BOW + Naive Bayes — 5-fold F1: {f1_bow.mean():.3f} ± {f1_bow.std():.3f}")

vocab = bow_pipe[0].fit_transform(X)
print(f"vocabulary: {vocab.shape[1]:,} terms over {vocab.shape[0]:,} docs")

BOW + Naive Bayes — 5-fold F1: 0.945 ± 0.008
vocabulary: 8,444 terms over 5,574 docs


## 5. TF-IDF: Downweighting the Boring Words

BOW counts treat *free* (rare, spammy) and *message* (everywhere) equally. **TF-IDF** rescales each count:

$$
\text{tfidf}(t, d) = \text{tf}(t, d) \times \log\frac{N + 1}{\text{df}(t) + 1} + 1
$$

— term frequency times **inverse document frequency**: a word seen in nearly every document gets idf ≈ 0 (uninformative); a word in a handful of documents gets a large idf. Check the learned weights on this corpus: *ok* (chatty, common) scores low; *win* and *claim* (spam-specific) score high.


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tfidf = TfidfVectorizer(stop_words=list(ENGLISH_STOP_WORDS))
tfidf.fit(X)
idf = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))
for w in ["ok", "free", "claim", "win", "urgent"]:
    print(f"  idf({w:8}) = {idf.get(w, float('nan')):.2f}")

tfidf_pipe = make_pipeline(
    TfidfVectorizer(stop_words=list(ENGLISH_STOP_WORDS)),
    LogisticRegression(max_iter=1000),
)
f1_tfidf = cross_val_score(tfidf_pipe, X, y, cv=5, scoring="f1")
print(f"\nTF-IDF + Logistic Regression — 5-fold F1: {f1_tfidf.mean():.3f} ± {f1_tfidf.std():.3f}")
print(f"accuracy trap: always predicting 'ham' alone scores {(1 - y.mean()):.1%}")

  idf(ok      ) = 3.99
  idf(free    ) = 4.19
  idf(claim   ) = 4.93
  idf(win     ) = 5.36
  idf(urgent  ) = 5.38



TF-IDF + Logistic Regression — 5-fold F1: 0.847 ± 0.021
accuracy trap: always predicting 'ham' alone scores 86.6%


## 6. Word Embeddings: Meaning as Geometry

BOW treats *prize* and *award* as unrelated columns. **Word2Vec** (Mikolov et al., 2013) instead learns a dense vector per word by predicting context: a shallow network trained on either *skip-gram* (word → context) or *CBOW* (context → word). After training, the **distributional hypothesis** — *words in similar contexts have similar meanings* — becomes measurable geometry: cosine similarity between vectors, and even vector arithmetic ($\vec{win} + \vec{prize}$ lands near $\vec{cash}$).

We train Word2Vec **from scratch** on this small corpus (honest scale: ~1.8k vocabulary, seconds on CPU) — in practice you would load pretrained vectors or, better, contextual embeddings from a Transformer (Section 7).


In [5]:
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

tokenized = [simple_preprocess(m) for m in X]
w2v = Word2Vec(sentences=tokenized, vector_size=50, window=5, min_count=5,
               workers=4, seed=42, epochs=10)
print(f"trained {len(w2v.wv):,} vectors of dimension {w2v.wv.vector_size} in under a second\n")

for word in ["free", "win", "txt"]:
    sims = ", ".join(f"{w} ({s:.2f})" for w, s in w2v.wv.most_similar(word, topn=5))
    print(f"most similar to '{word}': {sims}\n")

print("vector arithmetic:  'win' + 'prize' ≈")
print(" ", [f"{w} ({s:.2f})" for w, s in w2v.wv.most_similar(positive=["win", "prize"], topn=3)])

trained 1,788 vectors of dimension 50 in under a second

most similar to 'free': nokia (0.98), mobile (0.98), tone (0.97), txt (0.97), reply (0.96)

most similar to 'win': entry (0.99), tone (0.99), weekly (0.98), custcare (0.98), wk (0.98)

most similar to 'txt': www (0.98), tone (0.98), reply (0.98), nokia (0.97), stop (0.97)

vector arithmetic:  'win' + 'prize' ≈
  ['guaranteed (0.99)', 'cash (0.98)', 'service (0.98)']


Notice what the model learned with **no labels at all**: the neighbors of *free* and *win* are the vocabulary of spam promotions — *reply, nokia, mobile, tone, entry, weekly*. Clustering those vectors would nearly reconstruct the spam/ham split. That is the power (and limit) of static embeddings: one vector per word regardless of context — *bank* (river) and *bank* (money) share one vector.

## 7. From Sequences to Transformers

**RNN/LSTM** (Level 7 prerequisites) read text token-by-token carrying a memory state — but gradients vanish over long distances and training cannot be parallelized across time. The **Transformer** (Vaswani et al., 2017) replaced recurrence with **self-attention**:

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$

— every token directly attends to every other token, in parallel. Stack attention layers, pretrain on massive corpora, and you get BERT (encoder — classification, search) and GPT (decoder — generation). Since ~2019 the winning move is **pretrain-then-fine-tune**, not training from scratch; by 2026 most teams prompt or lightly fine-tune LLMs instead. That whole story — attention mechanics, tokenization, fine-tuning, RAG, agents — is **Level 7** of this roadmap; the [Fundamentals of Neural Networks](https://github.com/AdilShamim8/ML-Roadmap-and-Notes/tree/main/4.%20Diving%20into%20different%20domains/Fundamentals%20of%20Neural%20Networks) note is the on-ramp.

## 8. The 2026 Landscape: When to Use What

| Task | 2026 default | Why |
|---|---|---|
| High-volume text classification (spam, routing) | **BOW/TF-IDF + linear model or NB** | Millisecond latency, no GPU, explainable; F1 ≈ 0.94 above |
| Production pipelines (NER, POS, parsing) | **spaCy** | Industrial-speed pipelines, pretrained statistical models |
| Domain-specific semantics | **Transformer fine-tune** (DistilBERT-class) or embeddings-as-features | One vector per *context*, not per word |
| Open-ended understanding/generation | **LLM APIs or open-weights models** (Level 7) | Prompting beats training when the task is broad |
| Search / RAG | Embedding models + vector DB | Retrieval is the core of LLM apps — Level 7 M8 |

A disciplined 2026 practitioner reaches for the **cheapest tool that clears the bar** — the staircase of Section 1, climbed only as far as the task requires.


## Key Takeaways

- NLP's staircase: **preprocess → BOW/TF-IDF → embeddings → RNN → Transformers → LLMs**; classic rungs still win on speed/cost for high-volume classification.
- Put **preprocessing inside the pipeline** (vectorizer + model), so cross-validation is leak-free (Level 3 §6/§16).
- **TF-IDF** = counts × rarity; check `idf_` to see which words your corpus actually values.
- **Word2Vec** turns the distributional hypothesis into cosine geometry — trained from scratch here in seconds; use pretrained/contextual embeddings in production.
- Report **F1/precision-recall**, not accuracy, on imbalanced text tasks (86.6% "ham-only" accuracy is worthless — Level 2 §7).
- Continue: the [NLP-Introduction notebook on Kaggle](https://www.kaggle.com/code/adilshamim8/nlp-introduction) for a first applied pass, the [Email Spam Classifier project](https://github.com/AdilShamim8/Email-Spam-Classifier) for a full build, and **Level 7** for Transformers, LLMs, and RAG.
